In [ ]:
from pathlib import Path
import subprocess
import json
import shutil
import time
from datetime import datetime

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1]

PDF_POLICY_DIR = PROJECT_ROOT / "pdfs" / "policy"
MARKDOWN_POLICY_DIR = PROJECT_ROOT / "markdown" / "raw" / "policy"
LOG_DIR = PROJECT_ROOT / "progress" / "markdown_extraction" / "logs"

COUNTRIES = ["australia", "france", "ireland", "usa"]

print("Project root:", PROJECT_ROOT)
print("PDF policy dir:", PDF_POLICY_DIR)
print("Markdown policy dir:", MARKDOWN_POLICY_DIR)
print("Log dir:", LOG_DIR)

In [ ]:
def check_docling():
    result = subprocess.run(
        ["docling", "--version"],
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        raise RuntimeError(
            "Docling is not available in this environment. "
            "Activate the correct virtual environment or install it with: pip install docling"
        )
    
    print("Docling is available:")
    print(result.stdout.strip() or result.stderr.strip())

check_docling()

In [ ]:
for country in COUNTRIES:
    (MARKDOWN_POLICY_DIR / country).mkdir(parents=True, exist_ok=True)

LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Output folders are ready.")

In [ ]:
def find_policy_pdfs():
    pdf_records = []
    
    for country in COUNTRIES:
        country_pdf_dir = PDF_POLICY_DIR / country
        
        if not country_pdf_dir.exists():
            print(f"Warning: missing folder: {country_pdf_dir}")
            continue
        
        pdf_files = sorted(country_pdf_dir.glob("*.pdf"))
        
        for pdf_path in pdf_files:
            pdf_records.append({
                "country": country,
                "pdf_path": pdf_path,
                "output_dir": MARKDOWN_POLICY_DIR / country
            })
    
    return pdf_records


pdf_records = find_policy_pdfs()

print(f"Found {len(pdf_records)} PDF files.")
for item in pdf_records:
    print(item["country"], "->", item["pdf_path"].name)

In [ ]:
def convert_pdf_to_markdown(
    pdf_path: Path,
    output_dir: Path,
    *,
    overwrite: bool = False,
    enrich_pictures: bool = False,
    enrich_charts: bool = False,
    device: str = "cpu",
    timeout_seconds: int = 900
):
    """
    Convert a single PDF to Markdown using Docling.

    Parameters
    ----------
    pdf_path:
        Input PDF path.
    output_dir:
        Output directory for the generated Markdown.
    overwrite:
        If False, skip conversion when a likely Markdown file already exists.
    enrich_pictures:
        If True, enable Docling picture description.
        This may be slower and may require model artifacts depending on your setup.
    enrich_charts:
        If True, enable chart extraction.
    device:
        Use "cpu" for stable conversion on normal laptop/desktop.
        Use "cuda" only if you have a working NVIDIA GPU environment.
    timeout_seconds:
        Timeout per document.
    """

    output_dir.mkdir(parents=True, exist_ok=True)

    expected_md = output_dir / f"{pdf_path.stem}.md"

    if expected_md.exists() and not overwrite:
        return {
            "pdf": str(pdf_path),
            "output_dir": str(output_dir),
            "status": "skipped_exists",
            "markdown": str(expected_md),
            "returncode": 0,
            "stdout": "",
            "stderr": ""
        }

    cmd = [
        "docling",
        str(pdf_path),
        "--from", "pdf",
        "--to", "md",
        "--output", str(output_dir),
        "--image-export-mode", "referenced",
        "--ocr",
        "--tables",
        "--table-mode", "accurate",
        "--device", device,
        "--document-timeout", str(timeout_seconds),
        "--num-threads", "4",
    ]

    if enrich_pictures:
        cmd.append("--enrich-picture-description")

    if enrich_charts:
        cmd.append("--enrich-chart-extraction")

    started = time.time()

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    elapsed = round(time.time() - started, 2)

    status = "success" if result.returncode == 0 else "failed"

    return {
        "pdf": str(pdf_path),
        "output_dir": str(output_dir),
        "status": status,
        "markdown": str(expected_md) if expected_md.exists() else None,
        "returncode": result.returncode,
        "elapsed_seconds": elapsed,
        "stdout": result.stdout,
        "stderr": result.stderr,
        "command": " ".join(cmd)
    }

In [ ]:
conversion_logs = []

for index, record in enumerate(pdf_records, start=1):
    country = record["country"]
    pdf_path = record["pdf_path"]
    output_dir = record["output_dir"]

    print(f"[{index}/{len(pdf_records)}] Converting [{country}]: {pdf_path.name}")

    log = convert_pdf_to_markdown(
        pdf_path=pdf_path,
        output_dir=output_dir,
        overwrite=False,
        enrich_pictures=False,
        enrich_charts=False,
        device="cpu",
        timeout_seconds=900
    )

    conversion_logs.append(log)

    print("  status:", log["status"])
    if log["status"] == "failed":
        print("  stderr:", log["stderr"][:1000])

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = LOG_DIR / f"docling_policy_conversion_{timestamp}.json"

with open(log_file, "w", encoding="utf-8") as f:
    json.dump(conversion_logs, f, indent=2, ensure_ascii=False)

print("Saved log to:", log_file)

In [ ]:
from collections import Counter

status_counts = Counter(item["status"] for item in conversion_logs)

print("Conversion summary:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

failed = [item for item in conversion_logs if item["status"] == "failed"]

if failed:
    print("\nFailed files:")
    for item in failed:
        print("-", item["pdf"])

In [ ]:
generated_markdowns = sorted(MARKDOWN_POLICY_DIR.glob("*/*.md"))

print(f"Generated Markdown files: {len(generated_markdowns)}")

for md_path in generated_markdowns:
    print(md_path.relative_to(PROJECT_ROOT))